# Data Preparation for Indic ASR

This notebook handles dataset preparation for Common Voice and custom datasets.

**Supported Formats:**
- Common Voice 17.0 (Mozilla)
- IndicVoices Dataset
- Custom CSV/TSV with audio paths

In [ ]:
import subprocess
import sys

# Install dependencies
packages = ["datasets>=2.13.0", "librosa>=0.10.0", "soundfile>=0.12.0"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

from datasets import load_dataset, DatasetDict, Dataset
import pandas as pd
import os

print("✓ Dependencies installed")

In [ ]:
# Load Common Voice dataset
LANGUAGE = "mr"  # Change to "gu" or "hi"

print(f"Loading Common Voice for {LANGUAGE}...")
common_voice = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    LANGUAGE,
    trust_remote_code=True
)

print(f"Dataset Info:")
print(f"  Splits: {list(common_voice.keys())}")
for split, dataset in common_voice.items():
    print(f"  {split}: {len(dataset)} samples")

# Examine sample
sample = common_voice["train"][0]
print(f"\nSample structure:")
for key, value in sample.items():
    if key == "audio":
        print(f"  {key}: {type(value)} - sampling_rate={value['sampling_rate']}")
    elif isinstance(value, (str, int, float)):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {type(value)}")

In [ ]:
# Create balanced subsets for training
def create_balanced_subsets(dataset_dict, train_size=500, val_size=100, test_size=100):
    """Create balanced train/val/test splits."""
    
    subsets = {}
    
    for split_name in ["train", "test"]:
        if split_name in dataset_dict:
            original = dataset_dict[split_name]
            
            if split_name == "train":
                subsets["train"] = original.select(range(min(train_size, len(original))))
                if len(original) > train_size:
                    remaining = original.select(range(train_size, len(original)))
                    subsets["validation"] = remaining.select(range(min(val_size, len(remaining))))
            else:
                subsets["test"] = original.select(range(min(test_size, len(original))))
    
    return DatasetDict(subsets)

# Create subsets
balanced_splits = create_balanced_subsets(common_voice, train_size=500, val_size=100, test_size=100)

print("Created balanced splits:")
for split, dataset in balanced_splits.items():
    print(f"  {split}: {len(dataset)} samples")

# Save to disk for reuse
output_dir = "/tmp/indic_cv_subsets"
os.makedirs(output_dir, exist_ok=True)
balanced_splits.save_to_disk(f"{output_dir}/{LANGUAGE}")
print(f"✓ Saved to {output_dir}/{LANGUAGE}")